In [2]:
class Resistor(object):
    def __init__(self, ohms):
        self.ohms = ohms
        self.voltage = 0
        self.current = 0
        
r1 = Resistor(50e3)
r1.ohms = 10e3


class VoltageResistance(Resistor):
    def __init__(self, ohms):
        super().__init__(ohms)
        self._voltage = 0
        
    @property
    def voltage(self):
        return self._voltage
    
    @voltage.setter
    def voltage(self, voltage):
        self._voltage = voltage
        self.current = self._voltage /self.ohms
        
r2 = VoltageResistance(1e3)
print('Before :%5r amps' % r2.current)
r2.voltage = 10
print('After : %5r amps'% r2.current)

Before :    0 amps
After :  0.01 amps


In [3]:
class BoundedResistance(Resistor):
    def __init__(self, ohms):
        super().__init__(ohms)
        
    @property
    def ohms(self):
        return self._ohms

    @ohms.setter
    def ohms(self, ohms):
        if ohms <= 0:
            raise ValueError('%f ohms must be > 0' % ohms)
        self._ohms = ohms
    
r3 =BoundedResistance(1e3)
r3.ohms = 0

ValueError: 0.000000 ohms must be > 0

In [5]:
class FixedResistance(Resistor):
    def __init__(self, ohms):
        super().__init__(ohms)
        
    @property
    def ohms(self):
        return self._ohms

    @ohms.setter
    def ohms(self, ohms):
        if hasattr(self, '_ohms'):
            raise AttributeError("Can't set attribute")
        self._ohms=ohms
        
r4 = FixedResistance(1e3)
r4.ohms = 2e3


AttributeError: Can't set attribute

In [11]:
import time
from datetime import datetime, timedelta

class Bucket(object):
    def __init__(self, period):
        self.period_data = timedelta(seconds=period)
        self.reset_time = datetime.now()
        self.quota = 0
        
    def __repr__(self):
        return 'Bucket(quota=%d)' % self.quota
        
    
    # 채우는 동작의 구현
    def fill(bucket, amount):
        now = datetime.now()
        if now - bucket.reset_time > bucket.period_data:
            bucket.quota = 0
            bucket.reset_time = now
        bucket.quota += amount
    # 할당량 소비의 구현
    def deduct(bucket, amount):
        now = datetime.now()
        if now - bucket.reset_time > bucket.period_data:
            return False
        if bucket.quota - amount < 0 :
            return False
        bucket.quota -= amount
        return True
 # 양동이 채우기   
bucket = Bucket(60)
bucket.fill(100)
print(bucket)

if bucket.deduct(99):
    print('Had 99 quota')
else:
    print('Not enough for 99 quota')
print(bucket)    

if bucket.deduct(3):
    print('Had 3 quota')
else:
    print('Not enough for 3 quota')
print(bucket)    

Bucket(quota=100)
Had 99 quota
Bucket(quota=1)
Not enough for 3 quota
Bucket(quota=1)


In [13]:
import time
from datetime import datetime, timedelta

class Bucket(object):
    def __init__(self, period):
        self.period_data = timedelta(seconds=period)
        self.reset_time = datetime.now()
        self.max_quota = 0
        self.quota_consumed = 0
        
    def __repr__(self):
        return 'Bucket(quota=%d, qutoa consumed=%d)' % (self.max_quota, self.quota_consumed)
    
    @property
    def quota(self):
        return self.max_quota - self.quota_consumed
    
    @quota.setter
    def quota(self, amount):
        delta = self.max_quota - amount
        if amount == 0:
            #새 기간의 할당량을 리셋함
            self.quota_consumed = 0
            self.max_quota = 0
        elif delta < 0:
            # 새 기간의 할당량을 채움
            assert self.quota_consumed == 0
            self.max_quota = amount
        else:
            # 기간동안 할당량의 소비
            assert self.max_quota >= self.quota_consumed
            self.quota_consumed += delta
            
    def fill(bucket, amount):
        now = datetime.now()
        if now - bucket.reset_time > bucket.period_data:
            bucket.quota = 0
            bucket.reset_time = now
        bucket.quota += amount
    # 할당량 소비의 구현
    def deduct(bucket, amount):
        now = datetime.now()
        if now - bucket.reset_time > bucket.period_data:
            return False
        if bucket.quota - amount < 0 :
            return False
        bucket.quota -= amount
        return True
        
    

 # 양동이 채우기   
bucket = Bucket(60)
bucket.fill(100)
print(bucket)

if bucket.deduct(99):
    print('Had 99 quota')
else:
    print('Not enough for 99 quota')
print(bucket)    

if bucket.deduct(3):
    print('Had 3 quota')
else:
    print('Not enough for 3 quota')
print(bucket)    

Bucket(quota=100, qutoa consumed=0)
Had 99 quota
Bucket(quota=100, qutoa consumed=99)
Not enough for 3 quota
Bucket(quota=100, qutoa consumed=99)


In [15]:
class Homework(object):
    def __init__(self):
        self._grade = 0
    
    @property
    def grade(self):
        return self._grade

    @grade.setter
    def grade(self, value):
        if not(0 <= value <= 100):
            raise ValueError('Grade must be between 0 and 100')
        self._grade = value
        
galic = Homework()
galic.grade = 95
print(galic)


In [20]:
from weakref import WeakKeyDictionary

class Grade(object):
    def __init__(self):
        self._value = WeakKeyDictionary()
        
    def __get__(self, instance, instance_type):
        if instance is None : return self
        return self._value.get(instance, 0)
    
    def __set__(self, instance, value):
        if not (0 <= value <= 100):
            raise ValueError('Grade must be between 0 and 100')
        self._value[instance] = value

class Exam(object):
    math_grade = Grade()
    writing_grade = Grade()
    science_grade = Grade()
        
first_exam = Exam()
first_exam.writing_grade = 82
first_exam.science_grade = 99
print(first_exam.writing_grade,first_exam.science_grade )

second_exam = Exam()
second_exam.writing_grade = 75
print(first_exam.writing_grade,second_exam.writing_grade )

82 99
82 75


In [ ]:
class LazyDB(object):
    def __init__(self):
        self.exists = 5
        
    def __getattr__(self, name):
        value = 'Value for %s' % name
        setattr(self, name, value)
        return value

    
data = LazyDB()
print('Before:', data.__dict__)
print('foo: ', data.foo)
print('After: ',data.__dict__)

Before: {'exists': 5}
foo:  Value for foo
After:  {'exists': 5, 'foo': 'Value for foo'}


In [5]:
class ValidatingDB(object):
    def __init__(self):
        self.exists = 5
        
    def __getattribute__(self, name):
        print('Called __getattribute__(%s)' % name)
        try:
            return super().__getattribute__(name)
        except AttributeError:
            value = 'working __getattribute__ for %s' % name
            setattr(self,name,value)
            return value
        
data = ValidatingDB()
print('exists:', data.exists)
print('foo:', data.foo)
print('foo:', data.foo)


Called __getattribute__(exists)
exists: 5
Called __getattribute__(foo)
foo: working __getattribute__ for foo
Called __getattribute__(foo)
foo: working __getattribute__ for foo


In [6]:
class SavingDB():
    def __setattr__(self, name, value):
        ## DB에 저장하는코드
        
        super().__setattr__(name,value)
        
class LoggingSavingDB(SavingDB):
    def __setattr__(self,name,value):
        print('Called __setattr__(%s, %r) % (name,value)')
        super().__setattr__(name,value)
        
data = LoggingSavingDB()
print('Before: ', data.__dict__)
data.foo = 5
print('After: ', data.__dict__)
data.foo = 7
print('Finally: ', data.__dict__)

Before:  {}
Called __setattr__(%s, %r) % (name,value)
After:  {'foo': 5}
Called __setattr__(%s, %r) % (name,value)
Finally:  {'foo': 7}


In [11]:
class DictionaryDB(object):
    def __init__(self, data):
        self._data = data
        
    def __getattribute__(self, name):
        print('Called __getattribute__(%s)' % name)
        data_dict = super().__getattribute__('_data')
        return data_dict[name]
    
data = DictionaryDB({'foo': 3})
data.foo

Called __getattribute__(foo)


3

In [14]:
class Meta(type):
    def __new__(meta, name, bases, class_dict):
        print((meta, name, bases, class_dict))
        return type.__new__(meta, name, bases, class_dict)
    
class MyClass(object, metaclass =Meta):
    stuff = 123
    
    def foo(self):
        pass
    
class ValidatePolygon(type):
    def __new__(meta,name,bases, class_dict):
        # 추상 클래스는 검증하지 않는다.
        if bases != (object,):
            if class_dict['sides'] <3:
                raise ValueError('Polygons need 3+ sides')
        return type.__new__(meta,name,bases,class_dict)
    
class Polygon(object, metaclass =ValidatePolygon):
    sides = None
    
    @classmethod
    def inerior_angles(cls):
        return (cls.sides -2) * 180
    
class Triagnle(Polygon):
    sides = 3
    
print('Before class')
class Line(Polygon):
    print('Before sides')
    sides =1
    print('After sides')
print('After sides')

(<class '__main__.Meta'>, 'MyClass', (<class 'object'>,), {'__module__': '__main__', '__qualname__': 'MyClass', '__firstlineno__': 6, 'stuff': 123, 'foo': <function MyClass.foo at 0x0000020E26520040>, '__static_attributes__': (), '__classdictcell__': <cell at 0x0000020E265C2950: dict object at 0x0000020E264C9040>})
Before class
Before sides
After sides


ValueError: Polygons need 3+ sides

In [ ]:
import json

class Meta(type):
    def __new__(meta, name, bases, class_dict):
        cls = type.__new__(meta, name, bases, class_dict)
        register_class(cls)
        return cls
    


class Serializable():
    def __init__(self,*args):
        self.args = args
        
    def serialize(self):
        return json.dumps({'class': self.__class__.__name__,'args': self.args})
    
    def __repr__(self):
        return 'Point@D(%d, %d)' % (self.x, self.y)

registry = {}
class Regist(Serializable,metaclass= Meta):
    pass
    

def register_class(target_class):
    registry[target_class.__name__] = target_class
    
  
def deserialize(data):
    params = json.loads(data)
    name = params['class']
    target_class = registry[name]
    return target_class(*params['args'])
    
    
class Point2D(Regist):
    def __init__(self, x, y):
        super().__init__(x,y)
        self.x = x
        self.y = y



point = Point2D(5,3)
data = point.serialize()
after = deserialize(data)
print(point, data, after)

Point@D(5, 3) {"class": "Point2D", "args": [5, 3]} Point@D(5, 3)


In [ ]:

class Field(object):
    def __init__(self):
        self.name = None
        self.internal_name = None
    
    def __get__(self, instance, instance_type):
        if instance is None: return self
        return getattr(instance, self.internal_name,'')
    
    def __set__(self, instance, value):
        setattr(instance,self.internal_name, value)
class Meta(type):
    def __new__(meta,name,bases, class_dict):
        for key, value in class_dict.items():
            if isinstance(value, Field):
                value.name = key
                value.internal_name = '_'  + key
        cls = type.__new__(meta, name, bases, class_dict)
        return cls

class DatabaseRow(object, metaclass=Meta):
    pass
        
        
class Customer(DatabaseRow):
    first_name = Field()
    last_name = Field()
    prefix = Field()
    suffix = Field()
    

    
foo = Customer()
print(foo.__dict__)
foo.first_name = 'Euler'
print(foo.__dict__)

TypeError: Field.__init__() missing 1 required positional argument: 'name'